In [1]:
import os
import hydra
import torch
import dill
from omegaconf import OmegaConf
import pathlib
from torch.utils.data import DataLoader
import copy
import random
import wandb
import tqdm
import numpy as np
from termcolor import cprint
import shutil
import time
import threading
from hydra.core.hydra_config import HydraConfig
from diffusion_policy_3d.policy.dp3 import DP3, DP3Compliant, DP3PcdCompliant, DP3Realworld
from diffusion_policy_3d.dataset.base_dataset import BaseDataset
from diffusion_policy_3d.env_runner.base_runner import BaseRunner
from diffusion_policy_3d.common.checkpoint_util import TopKCheckpointManager
from diffusion_policy_3d.common.pytorch_util import dict_apply, optimizer_to
from diffusion_policy_3d.model.diffusion.ema_model import EMAModel
from diffusion_policy_3d.model.common.lr_scheduler import get_scheduler

from icecream import ic


OmegaConf.register_new_resolver("eval", eval, replace=True)

In [ ]:
@hydra.main(config_name="dp3_realworld.yaml",
            version_base=None,
            config_path=str(pathlib.Path(__file__).parent.joinpath(
                'diffusion_policy_3d', 'config'))
)
def main(cfg: DictConfig):
    ic()
    ic(cfg['task_name']) # TODO: find out where to input task yaml during rollout
    training_use_ema = False

    workspace = TrainDP3Workspace(cfg=cfg)

    # best_ckpt_path = workspace.get_checkpoint_path(tag="best")
    # ckpt_dir = '/home/mh2595/workspace/implicit_force_simulation/third_party/3D-Diffusion-Policy/3D-Diffusion-Policy/data/outputs/realworld_contact_29_5Hz-dp3_realworld_horizon1-rate_new_seed6/checkpoints'
    ckpt_dir = '/home/mh2595/workspace/implicit_force_simulation/third_party/3D-Diffusion-Policy/3D-Diffusion-Policy/data/outputs/realworld_contact_29_10Hz-dp3_realworld_horizon2-hor2_new_seed6'
    # ckpt_dir = '/home/mh2595/workspace/implicit_force_simulation/third_party/3D-Diffusion-Policy/3D-Diffusion-Policy/data/outputs/realworld_contact_10Hz-dp3_realworld-0003_seed0/checkpoints'
    best_ckpt_path = pathlib.Path(ckpt_dir).joinpath("latest.ckpt")
    if best_ckpt_path.is_file():
        print(f"Resuming from checkpoint {best_ckpt_path}")
        workspace.load_checkpoint(path=best_ckpt_path)

    env_runner = RealworldRunner(output_dir='./',
                                eval_episodes=10,
                                max_steps=600,
                                fps=30, # 30Hz
                                # fps=10, # 10Hz
                                # fps=5, # 5Hz
                                n_obs_steps=2,
                                n_action_steps=10,) # TODO: check if action steps is correct
    # assert isinstance(env_runner, BaseRunner) # TODO: why not instance

    policy = workspace.model
    if training_use_ema:
        policy = workspace.ema_model
    policy.eval()
    policy.cuda()

    runner_log = env_runner.run(policy=policy, use_force=cfg['policy']['use_force'])

    cprint(f"---------------- Eval Results --------------", 'magenta')
    for key, value in runner_log.items():
        if isinstance(value, float):
            cprint(f"{key}: {value:.4f}", 'magenta')

In [ ]:
import hydra
from omegaconf import OmegaConf
import pathlib

config_path = str(pathlib.Path(__file__).parent.joinpath('diffusion_policy_3d', 'config'))

@hydra.main(
    version_base=None,
    config_path=config_path,
    config_name="dp3_realworld_dummy"  # Assuming the main configuration file is named '_.yaml'
)
def create_cfg(cfg):
    print(OmegaConf.to_yaml(cfg))
    return cfg

cfg = create_cfg()

env_runner: BaseRunner
env_runner = hydra.utils.instantiate(
    cfg.task.env_runner,
    output_dir=self.output_dir)

if env_runner is not None:
    assert isinstance(env_runner, BaseRunner)

runner_log = env_runner.run(policy)